In [1]:
import numpy as np
import pandas as pd
import subprocess

In [2]:
# generate negative and postive control phenotypes

In [3]:
vcf_in = "/home/ecxu/public/1000Genomes/1000G_chr10_pruned.vcf.gz"
igsr_tsv = "/home/ecxu/public/1000Genomes/igsr_samples.tsv"

n = 200                        
seed = 42

pheno_mean = 0.0
pheno_sd = 1.0

rng = np.random.default_rng(seed)

In [4]:
# Get all sample IDs
vcf_samples = subprocess.check_output(
    ["bcftools", "query", "-l", vcf_in],
    text=True
).splitlines()
vcf_samples = [s.strip() for s in vcf_samples if s.strip()]
vcf_samples_set = set(vcf_samples)

print(f"VCF samples: {len(vcf_samples)}")

VCF samples: 2504


In [5]:
# Load metadata and subset to covariates of interest
meta = pd.read_csv(igsr_tsv, sep="\t", dtype=str)

meta = meta.rename(columns={
    "Sample name": "sample",
    "Sex": "sex",
    "Population code": "population_code"
})[["sample", "sex", "population_code"]]

# Keep only samples that are in the VCF
meta = meta[meta["sample"].isin(vcf_samples_set)].drop_duplicates("sample").copy()

print(f"Samples in IGSR ∩ VCF: {len(meta)}")

if len(meta) < n:
    raise ValueError(f"Requested n={n} but only {len(meta)} samples overlap IGSR and VCF.")


Samples in IGSR ∩ VCF: 2504


In [6]:
# Get a random subset of individuals to simulate data for 
subset = meta.sample(n=n, random_state=seed).copy()
subset = subset.sort_values("sample").reset_index(drop=True)

subset_samples = subset["sample"].tolist()
print(f"Subset size: {len(subset_samples)}")

# Write the sampled individuals to a text file
samples_txt = "sample_subset.txt"
pd.Series(subset_samples).to_csv(samples_txt, index=False, header=False)
print("Wrote:", samples_txt)

Subset size: 200
Wrote: sample_subset.txt


### DIVIDER ###

In [7]:
# Subset VCF
vcf_out = "genotype_subset.vcf.gz"

subprocess.check_call(["bcftools", "view", "-S", samples_txt, "-Oz", "-o", vcf_out, vcf_in])
subprocess.check_call(["tabix", "-p", "vcf", vcf_out])

print("Wrote:", vcf_out)

Wrote: genotype_subset.vcf.gz


In [8]:
# Simulate phenotype (negative control, normal distribution)
pheno = rng.normal(loc=pheno_mean, scale=pheno_sd, size=n)

phen_df = pd.DataFrame({
    "FID": subset["sample"].values,
    "IID": subset["sample"].values,
    "PHENO": pheno
})

phen_path = "negative_control.phen"
phen_df.to_csv(phen_path, sep="\t", index=False)
print("Wrote:", phen_path)

Wrote: negative_control.phen


In [9]:
phen_df

,FID,IID,PHENO
0,HG00101,HG00101,0.304717
1,HG00154,HG00154,-1.039984
2,HG00189,HG00189,0.750451
3,HG00244,HG00244,0.940565
4,HG00260,HG00260,-1.951035
...,...,...,...
195,NA20819,NA20819,-0.894727
196,NA20822,NA20822,0.643327
197,NA20852,NA20852,-0.394605
198,NA20868,NA20868,-0.005122


In [10]:
# Set parameters for haptools
k_causal = 5                 
beta_sd = 0.6                
heritability = 0.6           

# Define output file names
snplist_path = "positive_control.snplist"
out_pheno_haptools = "positive_control.pheno"   
out_phen = "positive_control.phen" 

In [11]:
# Describe causal SNPs
ids = subprocess.check_output(
    ["bcftools", "query", "-f", "%ID\n", vcf_in],
    text=True
).splitlines()
ids = [x.strip() for x in ids if x.strip() and x.strip() != "."]

causal_ids = rng.choice(ids, size=k_causal, replace=False)
betas = rng.normal(loc=0.0, scale=beta_sd, size=k_causal)

with open(snplist_path, "w") as f:
    for vid, b in zip(causal_ids, betas):
        f.write(f"{vid}\t{b}\n")

print("Wrote:", snplist_path)
print("Causal SNPs (ID, beta):")
for vid, b in zip(causal_ids, betas):
    print(" ", vid, b)

Wrote: positive_control.snplist
Causal SNPs (ID, beta):
  rs12247225 -0.029231041158070768
  rs10508215 -0.5059381621757226
  rs7893395 -0.7312878362541768
  rs4747692 -0.5268914201572504
  rs7897351 -0.20047406442048724


In [12]:
# Use haptools to simulate positive phenotype
cmd = [
    "haptools", "simphenotype",
    "--samples-file", samples_txt,           
    "--heritability", str(heritability),
    "--seed", str(seed),
    "--output", out_pheno_haptools,
    vcf_in,
    snplist_path
]
subprocess.check_call(cmd)
print("Wrote:", out_pheno_haptools)

[    INFO] Loading from .snplist (sim_phenotype.py:388)
[    INFO] Loading haplotype genotypes from VCF/BCF file (sim_phenotype.py:426)
[    INFO] Loading genotypes from 200 samples (genotypes.py:287)


Wrote: positive_control.pheno


[    INFO] Transposing genotype matrix of size (5, 200, 3) (genotypes.py:207)
[    INFO] QC-ing genotypes (sim_phenotype.py:431)
[    INFO] Simulating phenotypes (sim_phenotype.py:458)
[    INFO] Computing genetic component w/ 5 causal effects (sim_phenotype.py:198)
[    INFO] Adding environmental component 0.7635333282933835 for h^2 0.6 (sim_phenotype.py:228)
[    INFO] Writing phenotypes (sim_phenotype.py:462)


In [13]:
ph = pd.read_csv(out_pheno_haptools, sep="\t")
print("Columns:", ph.columns.tolist())
display(ph.head())

# Get sample ID
if "#IID" in ph.columns:
    iid_col = "#IID"
elif "IID" in ph.columns:
    iid_col = "IID"
else:
    iid_col = ph.columns[0]

# Get phenotype information
pheno_cols = [c for c in ph.columns if c != iid_col]
if len(pheno_cols) == 0:
    raise ValueError("No phenotype column found in haptools output.")
pheno_col = pheno_cols[0]   

# Convert to format we expect (matching our other .phen file)
phen_df_pos = pd.DataFrame({
    "FID": ph[iid_col].astype(str),
    "IID": ph[iid_col].astype(str),
    "PHENO": ph[pheno_col].astype(float)
})

phen_df_pos.to_csv(out_phen, sep="\t", index=False)
print("Wrote:", out_phen)

display(phen_df_pos.head())

Columns: ['#IID', 'rs12247225-rs10508215-rs7893395-rs4747692-rs7897351']


,#IID,rs12247225-rs10508215-rs7893395-rs4747692-rs7897351
0,HG00101,0.517493
1,HG00154,0.249754
2,HG00189,1.237683
3,HG00244,1.294411
4,HG00260,-1.773112


Wrote: positive_control.phen


,FID,IID,PHENO
0,HG00101,HG00101,0.517493
1,HG00154,HG00154,0.249754
2,HG00189,HG00189,1.237683
3,HG00244,HG00244,1.294411
4,HG00260,HG00260,-1.773112


In [14]:
# Sanity checking samples
expected = set(subset["sample"].astype(str).tolist())
observed = set(phen_df_pos["IID"].astype(str).tolist())
print("n in phenotype:", len(phen_df_pos))
print("All IDs match:", expected == observed)

n in phenotype: 200
All IDs match: True


In [15]:
# Encode sex codes (male=1, female=2)
sex_map = {
    "male": 1, "m": 1, "1": 1,
    "female": 2, "f": 2, "2": 2
}

sex_num = (
    subset["sex"]
    .astype(str).str.strip().str.lower()
    .map(sex_map)
    .fillna(0)
    .astype(int)
)

# Encode population codes
pop = subset["population_code"].astype(str).str.strip()
pop_dummies = pd.get_dummies(pop, prefix="pop")

cov_df = pd.concat(
    [
        pd.DataFrame({"FID": subset["sample"].values, "IID": subset["sample"].values}),
        pd.DataFrame({"sex": sex_num.values}),
        pop_dummies.reset_index(drop=True),
    ],
    axis=1
)

cov_path = "samples.covar"
cov_df.to_csv(cov_path, sep="\t", index=False)
print("Wrote:", cov_path)

Wrote: samples.covar


In [16]:
# Sanity checking for cov
print("\nHead covar:")
display(cov_df.head())

vcf_n = int(subprocess.check_output(
    ["bash", "-lc", f"bcftools query -l {vcf_out} | wc -l"],
    text=True
).strip())
print(f"\nVCF sample count after subsetting: {vcf_n}")


Head covar:


,FID,IID,sex,pop_ACB,pop_ASW,pop_BEB,pop_CDX,pop_CEU,pop_CHB,pop_CHS,...,pop_KHV,pop_LWK,pop_MSL,pop_MXL,pop_PEL,pop_PJL,pop_PUR,pop_STU,pop_TSI,pop_YRI
0,HG00101,HG00101,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,HG00154,HG00154,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,HG00189,HG00189,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,HG00244,HG00244,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,HG00260,HG00260,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



VCF sample count after subsetting: 200


In [17]:
cov_df

,FID,IID,sex,pop_ACB,pop_ASW,pop_BEB,pop_CDX,pop_CEU,pop_CHB,pop_CHS,...,pop_KHV,pop_LWK,pop_MSL,pop_MXL,pop_PEL,pop_PJL,pop_PUR,pop_STU,pop_TSI,pop_YRI
0,HG00101,HG00101,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,HG00154,HG00154,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,HG00189,HG00189,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,HG00244,HG00244,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,HG00260,HG00260,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,NA20819,NA20819,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
196,NA20822,NA20822,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
197,NA20852,NA20852,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
198,NA20868,NA20868,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
